In [15]:
import pandas as pd 
import numpy as np
import re

from astroquery.simbad import Simbad

In [16]:
EXPLICIT_ALIASES = {
    'AUMic': ['AU Mic'],
    'EpsilonIndi': ['eps Ind', 'Epsilon Indi'],
    'piMensae': ['pi Men', 'Pi Mensae'],
}


def candidate_names(name):
    name = str(name).strip()
    candidates = [name]
    candidates.extend(EXPLICIT_ALIASES.get(name, []))

    patterns = [
        (r'^HD(\\d+)$', r'HD \\1'),
        (r'^HIP(\\d+)$', r'HIP \\1'),
        (r'^GJ(\\d+)$', r'GJ \\1'),
        (r'^(55)CncA$', r'55 Cnc'),
    ]

    for pattern, replacement in patterns:
        if re.match(pattern, name, flags=re.IGNORECASE):
            candidates.append(re.sub(pattern, replacement, name, flags=re.IGNORECASE))

    seen = set()
    ordered_candidates = []
    for candidate in candidates:
        if candidate not in seen:
            seen.add(candidate)
            ordered_candidates.append(candidate)

    return ordered_candidates


def fetch_gaia_dr3_id(name):
    for candidate in candidate_names(name):
        ids = Simbad.query_objectids(candidate)
        if ids is None:
            continue

        for row in ids:
            object_id = str(row[0])
            if object_id.startswith('Gaia DR3 '):
                return pd.Series({
                    'matched_name': candidate,
                    'gaia_dr3_id': object_id.replace('Gaia DR3 ', '')
                })

    return pd.Series({
        'matched_name': pd.NA,
        'gaia_dr3_id': pd.NA
    })


#stars_df[['matched_name', 'gaia_dr3_id']] = stars_df['star'].apply(fetch_gaia_dr3_id)

In [17]:
sample_df = pd.read_csv('sample.csv')
sample_df.head(5)

,star,matched_name,gaia_dr3_id
0,55CncA,55CncA,704967037090946688
1,AUMic,AU Mic,6794047652729201024
2,EpsilonIndi,eps Ind,6412595290592307840
3,GJ436,GJ436,4017860992519744384
4,HAT-P-26,HAT-P-26,3668036348641580288


In [18]:
sweetcat_url = 'https://sweetcat.iastro.pt/catalog/SWEETCAT_Dataframe.csv'
sweetcat_df = pd.read_csv(sweetcat_url)

# Keep Gaia IDs numeric while still allowing missing values.
sample_df['gaia_dr3_id'] = pd.to_numeric(sample_df['gaia_dr3_id'], errors='coerce').astype('Int64')
sweetcat_df['gaia_dr3'] = pd.to_numeric(sweetcat_df['gaia_dr3'], errors='coerce').astype('Int64')

sweetcat_columns = [
    'Name', 'gaia_dr3', 'Teff', 'eTeff', 'Logg', 'eLogg', '[Fe/H]', 'e[Fe/H]',
    'Vt', 'eVt', 'Gmag', 'Plx', 'Distance', 'Mass_t', 'Radius_t', 'SWFlag',
    'Reference', 'Link', 'Update', 'Database'
]

sample_sweetcat_df = sample_df.merge(
    sweetcat_df[sweetcat_columns],
    left_on='gaia_dr3_id',
    right_on='gaia_dr3',
    how='left'
)

sample_sweetcat_df.head()

,star,matched_name,gaia_dr3_id,Name,gaia_dr3,Teff,eTeff,Logg,eLogg,[Fe/H],...,Gmag,Plx,Distance,Mass_t,Radius_t,SWFlag,Reference,Link,Update,Database
0,55CncA,55CncA,704967037090946688,55 Cnc,704967037090946688,5363.0,59.0,4.29,0.14,0.33,...,5.732700,79.4482,12.586818,0.922748,0.901203,1,Sousa et al. 2021,https://ui.adsabs.harvard.edu/abs/2021arXiv210...,2021-01-01,EU;NASA
1,AUMic,AU Mic,6794047652729201024,AU Mic,6794047652729201024,3700.0,100.0,NaN,NaN,NaN,...,7.843400,102.9432,9.714095,0.512628,0.697741,0,Plavchan et al.2020,https://ui.adsabs.harvard.edu/abs/2020Natur.58...,2020-07-08,EU;NASA
2,EpsilonIndi,eps Ind,6412595290592307840,eps Ind A,6412595290592307840,4694.0,115.0,4.40,0.33,-0.17,...,4.322900,274.8431,3.638440,0.695643,0.883542,1,Sousa et al. 2021,https://ui.adsabs.harvard.edu/abs/2021arXiv210...,2021-01-01,EU;NASA
3,GJ436,GJ436,4017860992519744384,GJ 436,4017860992519744384,3478.0,91.0,NaN,NaN,-0.08,...,9.582008,102.3014,9.775037,0.417588,0.384887,1,Antoniadis-Karnavas et al. 2024,https://ui.adsabs.harvard.edu/abs/2024arXiv240...,2024-06-26,EU;NASA
4,HAT-P-26,HAT-P-26,3668036348641580288,HAT-P-26,3668036348641580288,5001.0,61.0,4.21,0.15,0.01,...,11.461800,6.9985,142.887762,0.779003,0.844307,1,Sousa et al. 2021,https://ui.adsabs.harvard.edu/abs/2021arXiv210...,2021-01-01,EU;NASA


In [24]:
selected_columns =['star', 'Name', 'gaia_dr3', 'Teff',
       'eTeff', 'Logg', 'eLogg', '[Fe/H]', 'e[Fe/H]', 'Vt', 'eVt', 'Gmag',
       'Plx', 'Distance', 'Mass_t', 'Radius_t', 'SWFlag', 'Reference', 'Link',
       'Update', 'Database']

In [27]:
sample_sweetcat_df = sample_sweetcat_df[selected_columns]
sample_sweetcat_df.to_csv('sample_sweetcat.csv', index=False)